# ManiSkill Mechanical Arm Rendering on Colab

This notebook is for rendering the simulated Panda arm in ManiSkill/SAPIEN and exporting MP4 videos. Use a GPU runtime first: **Runtime > Change runtime type > GPU**.


## 1. Check GPU and Vulkan

SAPIEN rendering needs Vulkan. If this section cannot find an NVIDIA GPU or Vulkan driver, restart Colab with a GPU runtime or reconnect to a new runtime.


In [3]:
!nvidia-smi
!apt-get update -qq
!apt-get install -y -qq vulkan-tools libvulkan1 ffmpeg
!ls /usr/share/vulkan/icd.d || true
!vulkaninfo --summary || true


Sat May  2 16:25:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Pull or update the project


In [4]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
!mkdir -p Robotics_Final
%cd Robotics_Final

import os
repo = "Fanal_project_of_Robotics"
if not os.path.exists(repo):
    !git clone https://github.com/Leo0902110/Fanal_project_of_Robotics.git
else:
    %cd Fanal_project_of_Robotics
    !git pull
    %cd ..

%cd Fanal_project_of_Robotics


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive
/content/drive/MyDrive/Robotics_Final
/content/drive/MyDrive/Robotics_Final/Fanal_project_of_Robotics
Already up to date.
/content/drive/MyDrive/Robotics_Final
/content/drive/MyDrive/Robotics_Final/Fanal_project_of_Robotics


## 3. Install dependencies

`requirements-A.txt` includes ManiSkill/SAPIEN, Open3D, and `pin` for Panda end-effector control.


In [5]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -r requirements-A.txt


## 4. Select the Vulkan ICD

Colab GPU runtimes normally expose an NVIDIA Vulkan ICD. This cell sets `VK_ICD_FILENAMES` when it can find one.


In [6]:
import glob, os

icds = glob.glob('/usr/share/vulkan/icd.d/*.json')
print('ICD files:', icds)
nvidia_icds = [p for p in icds if 'nvidia' in p.lower()]
if nvidia_icds:
    os.environ['VK_ICD_FILENAMES'] = nvidia_icds[0]
    print('Using VK_ICD_FILENAMES =', os.environ['VK_ICD_FILENAMES'])
else:
    print('No NVIDIA ICD found. Continuing without VK_ICD_FILENAMES.')


ICD files: ['/usr/share/vulkan/icd.d/lvp_icd.x86_64.json', '/usr/share/vulkan/icd.d/radeon_icd.x86_64.json', '/usr/share/vulkan/icd.d/intel_icd.x86_64.json', '/usr/share/vulkan/icd.d/virtio_icd.x86_64.json', '/usr/share/vulkan/icd.d/intel_hasvk_icd.x86_64.json']
No NVIDIA ICD found. Continuing without VK_ICD_FILENAMES.


## 5. Smoke tests

Run state first, then RGBD. The RGBD smoke test checks whether camera observations work before video rendering.


In [7]:
!python main.py --mode smoke --obs-mode state --max-steps 30 --no-video --output-dir results/colab_smoke_state
!python main.py --mode smoke --obs-mode rgbd --policy scripted --max-steps 30 --no-video --output-dir results/colab_smoke_rgbd



Running experiment: smoke_clean
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
环境 PickCube-v1 初始化成功，obs_mode=state
/usr/local/lib/python3.12/dist-packages/sapien/wrapper/pinocchio_model.py:254: UserWarning: Deprecated member. Use Frame.parentJoint instead.
  joint = self.model.frames[frame].parent
{
  "experiment": "smoke_clean",
  "condition": "Smoke",
  "env_id": "PickCube-v1",
  "obs_mode": "state",
  "pseudo_blur": false,
  "active_probe": false,
  "requested_policy": "scripted",
  "ef

## 6. Render active-probe mechanical arm video

Important: do **not** add `--no-video` here. The runner uses `render_mode='rgb_array'` and saves an MP4 into the output directory.


In [8]:
!python main.py \
  --mode mvp \
  --scene pseudo_blur \
  --policy scripted \
  --use-active-probe \
  --obs-mode rgbd \
  --max-steps 120 \
  --seed 42 \
  --output-dir results/colab_render_active_probe



Running experiment: active_probe
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
环境 PickCube-v1 初始化成功，obs_mode=rgbd
/usr/local/lib/python3.12/dist-packages/sapien/wrapper/pinocchio_model.py:254: UserWarning: Deprecated member. Use Frame.parentJoint instead.
  joint = self.model.frames[frame].parent
视频已存至: results/colab_render_active_probe/active_probe.mp4
{
  "experiment": "active_probe",
  "condition": "Active-Probe",
  "env_id": "PickCube-v1",
  "obs_mode": "rgbd",
  "pseudo_blur": true,


## 7. Display the rendered video


In [9]:
from pathlib import Path
from IPython.display import Video, display

out_dir = Path('results/colab_render_active_probe')
print('\n'.join(str(p) for p in sorted(out_dir.glob('*'))))

videos = sorted(out_dir.glob('*.mp4'))
if videos:
    display(Video(str(videos[0]), embed=True))
else:
    print('No MP4 found. Check whether render_mode failed or --no-video was accidentally used.')


results/colab_render_active_probe/active_probe.mp4
results/colab_render_active_probe/active_probe_decision_trace.csv
results/colab_render_active_probe/mvp_results.csv
results/colab_render_active_probe/mvp_results.json


## 8. Render the active-perception decision trace

This creates a second MP4 for the ambiguity/probe/refinement decision process.


In [10]:
!python scripts/render_decision_trace.py \
  --input results/colab_render_active_probe/active_probe_decision_trace.csv \
  --output results/active_probe_decision_trace.mp4

from IPython.display import Video, display
display(Video('results/active_probe_decision_trace.mp4', embed=True))


Saved trace animation: results/active_probe_decision_trace.mp4


## 9. Render a full algorithm demo

This combines the ManiSkill arm video with the active-perception trace, so the final MP4 shows the whole pipeline: visual uncertainty, tactile probing, boundary refinement, and grasp execution.


In [ ]:
!python scripts/render_full_algorithm_demo.py \
  --video results/colab_render_active_probe/active_probe.mp4 \
  --trace results/colab_render_active_probe/active_probe_decision_trace.csv \
  --output results/full_active_perception_demo.mp4

from IPython.display import Video, display
display(Video('results/full_active_perception_demo.mp4', embed=True))


## 10. Fallback if rendering fails

If Vulkan/RGBD rendering fails, the simulation may still work. Use this command for metrics only, then render on a different Colab runtime or your Mac M4 local setup.


In [11]:
!python main.py --mode mvp --scene pseudo_blur --policy scripted --use-active-probe --obs-mode state --max-steps 120 --seed 42 --no-video --output-dir results/colab_state_fallback



Running experiment: active_probe
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
环境 PickCube-v1 初始化成功，obs_mode=state
/usr/local/lib/python3.12/dist-packages/sapien/wrapper/pinocchio_model.py:254: UserWarning: Deprecated member. Use Frame.parentJoint instead.
  joint = self.model.frames[frame].parent
{
  "experiment": "active_probe",
  "condition": "Active-Probe",
  "env_id": "PickCube-v1",
  "obs_mode": "state",
  "pseudo_blur": true,
  "active_probe": true,
  "requested_policy": "scripted"

In [ ]:
!python scripts/collect_demos.py --num-episodes 20 --scene pseudo_blur --policy scripted --use-active-probe --obs-mode rgbd --max-steps 120 --seed 42 --output-dir data/demos/pickcube_active_probe

!python scripts/train_bc.py --demo-dir data/demos/pickcube_active_probe --output-dir runs/bc_active_probe --epochs 80 --batch-size 64 --hidden-dim 128 --device cuda

!python scripts/evaluate_bc.py --checkpoint runs/bc_active_probe/bc_policy.pt --num-episodes 10 --scene pseudo_blur --use-active-probe --obs-mode rgbd --max-steps 120 --seed 100 --output-dir results/bc_active_probe_eval --device cuda

!python scripts/render_decision_trace.py --input results/bc_active_probe_eval/episode_0000_bc_eval_trace.csv --output results/bc_active_probe_decision_trace.mp4


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
环境 PickCube-v1 初始化失败：'NoneType' object has no attribute 'visibility'
回退至 PickCube-v1 + state + 无渲染模式，确保 Colab smoke test 能继续。
{
  "path": "data/demos/pickcube_active_probe/episode_0000.npz",
  "episode_index": 0,
  "seed": 42,
  "env_id": "PickCube-v1",
  "requested_obs_mode": "rgbd",
  "obs_mode": "state",
  "scene": "pseudo_blur",
  "pseudo_blur": true,
  "active_probe": true,
  "requested_policy": "scripted",
  "effective_policy": "sine_fall